1. Install / Import Libraries

In [1]:
# If kagglehub is not installed in Colab, uncomment this:
# !pip install kagglehub

import os
import pandas as pd
import kagglehub

from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances

2. Download Dataset from Kaggle

In [2]:
# Download latest version of the dataset
dataset_path = kagglehub.dataset_download("mohdshahnawazaadil/restaurant-dataset")

print("Dataset downloaded at:", dataset_path)

100%|██████████| 433k/433k [00:00<00:00, 578kB/s]

Extracting files...
Dataset downloaded at: /root/.cache/kagglehub/datasets/mohdshahnawazaadil/restaurant-dataset/versions/1


3. Load CSV File

In [24]:
# Find CSV or TSV file inside the downloaded folder
files_in_dir = os.listdir(dataset_path)
print("Files in dataset folder:", files_in_dir)

data_file = None
for file_name in files_in_dir:
    if file_name.endswith(".csv") or file_name.endswith(".tsv"):
        data_file = file_name
        break

if data_file is None:
    raise FileNotFoundError(f"No CSV or TSV file found in: {dataset_path}")

full_file_path = os.path.join(dataset_path, data_file)

# Load dataset
if data_file.endswith(".csv"):
    df = pd.read_csv(full_file_path)
else:
    df = pd.read_csv(full_file_path, sep="\t")

print(f"Successfully loaded: {data_file}")
df.head()

Files in dataset folder: ['Dataset .csv']
Successfully loaded: Dataset .csv


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


4. Select Important Columns

In [4]:
important_cols = [
    'Restaurant Name',
    'City',
    'Cuisines',
    'Average Cost for two',
    'Price range',
    'Aggregate rating',
    'Votes'
]

df = df[important_cols].copy()
df.head()

,Restaurant Name,City,Cuisines,Average Cost for two,Price range,Aggregate rating,Votes
0,Le Petit Souffle,Makati City,"French, Japanese, Desserts",1100,3,4.8,314
1,Izakaya Kikufuji,Makati City,Japanese,1200,3,4.5,591
2,Heat - Edsa Shangri-La,Mandaluyong City,"Seafood, Asian, Filipino, Indian",4000,4,4.4,270
3,Ooma,Mandaluyong City,"Japanese, Sushi",1500,4,4.9,365
4,Sambo Kojin,Mandaluyong City,"Japanese, Korean",1500,4,4.8,229


5. Basic Preprocessing

In [5]:
# Remove rows with missing values
df.dropna(inplace=True)

# Convert cuisines string into list
df['Cuisines List'] = df['Cuisines'].apply(lambda x: [c.strip() for c in x.split(',')])

df.head()

,Restaurant Name,City,Cuisines,Average Cost for two,Price range,Aggregate rating,Votes,Cuisines List
0,Le Petit Souffle,Makati City,"French, Japanese, Desserts",1100,3,4.8,314,"[French, Japanese, Desserts]"
1,Izakaya Kikufuji,Makati City,Japanese,1200,3,4.5,591,[Japanese]
2,Heat - Edsa Shangri-La,Mandaluyong City,"Seafood, Asian, Filipino, Indian",4000,4,4.4,270,"[Seafood, Asian, Filipino, Indian]"
3,Ooma,Mandaluyong City,"Japanese, Sushi",1500,4,4.9,365,"[Japanese, Sushi]"
4,Sambo Kojin,Mandaluyong City,"Japanese, Korean",1500,4,4.8,229,"[Japanese, Korean]"


6. Convert Cuisines into One-Hot Encoded Columns

In [6]:
mlb = MultiLabelBinarizer()

cuisine_encoded = pd.DataFrame(
    mlb.fit_transform(df['Cuisines List']),
    index=df.index,
    columns=mlb.classes_
)

cuisine_encoded.head()

,Afghani,African,American,Andhra,Arabian,Argentine,Armenian,Asian,Asian Fusion,Assamese,...,Teriyaki,Tex-Mex,Thai,Tibetan,Turkish,Turkish Pizza,Vegetarian,Vietnamese,Western,World Cuisine
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


7. Create Final Feature Dataset

In [7]:
# Drop columns not needed directly
df_features = df.drop(columns=['City', 'Cuisines', 'Cuisines List']).copy()

# Combine numeric features + cuisine encoded features
df_full = pd.concat([df_features, cuisine_encoded], axis=1)

# Remove any remaining missing values
df_full.dropna(inplace=True)

# Set restaurant name as index
df_full.set_index('Restaurant Name', inplace=True)

df_full.head()

,Average Cost for two,Price range,Aggregate rating,Votes,Afghani,African,American,Andhra,Arabian,Argentine,...,Teriyaki,Tex-Mex,Thai,Tibetan,Turkish,Turkish Pizza,Vegetarian,Vietnamese,Western,World Cuisine
Restaurant Name,,,,,,,,,,,,,,,,,,,,,
Le Petit Souffle,1100,3,4.8,314,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Izakaya Kikufuji,1200,3,4.5,591,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Heat - Edsa Shangri-La,4000,4,4.4,270,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Ooma,1500,4,4.9,365,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Sambo Kojin,1500,4,4.8,229,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


8. Check Final Data Shape

In [8]:
print("Final dataset shape:", df_full.shape)
df_full.head()

Final dataset shape: (9542, 149)


,Average Cost for two,Price range,Aggregate rating,Votes,Afghani,African,American,Andhra,Arabian,Argentine,...,Teriyaki,Tex-Mex,Thai,Tibetan,Turkish,Turkish Pizza,Vegetarian,Vietnamese,Western,World Cuisine
Restaurant Name,,,,,,,,,,,,,,,,,,,,,
Le Petit Souffle,1100,3,4.8,314,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Izakaya Kikufuji,1200,3,4.5,591,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Heat - Edsa Shangri-La,4000,4,4.4,270,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Ooma,1500,4,4.9,365,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Sambo Kojin,1500,4,4.8,229,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# 9. Cosine Similarity Without Scaling

In [9]:
cosine_matrix = cosine_similarity(df_full)

df_cosine = pd.DataFrame(
    cosine_matrix,
    index=df_full.index,
    columns=df_full.index
)

df_cosine.head()

Restaurant Name,Le Petit Souffle,Izakaya Kikufuji,Heat - Edsa Shangri-La,Ooma,Sambo Kojin,Din Tai Fung,Buffet 101,Vikings,Spiral - Sofitel Philippine Plaza Manila,Locavore,...,Emirgan S�_ti��,Leman K�_lt�_r,Dem Karak�_y,Karak�_y G�_ll�_o��lu,Baltazar,Naml۱ Gurme,Ceviz A��ac۱,Huqqa,A���k Kahve,Walter's Coffee Roastery
Restaurant Name,,,,,,,,,,,,,,,,,,,,,
Le Petit Souffle,1.000000,0.983920,0.977886,0.999224,0.992000,0.998936,0.999715,0.998828,0.984731,0.985173,...,0.355445,0.421308,0.318400,0.303835,0.371997,0.370229,0.370248,0.505366,0.399053,0.362434
Izakaya Kikufuji,0.983920,1.000000,0.924818,0.976129,0.953507,0.991104,0.979412,0.991399,0.937817,0.999974,...,0.516666,0.576503,0.482584,0.469108,0.531798,0.530184,0.530203,0.651352,0.556400,0.523060
Heat - Edsa Shangri-La,0.977886,0.924818,1.000000,0.985361,0.996462,0.967214,0.982571,0.966647,0.999361,0.927516,...,0.152122,0.222332,0.113121,0.097887,0.169660,0.167782,0.167805,0.313740,0.198482,0.159515
Ooma,0.999224,0.976129,0.985361,1.000000,0.996200,0.996350,0.999876,0.996155,0.990821,0.977658,...,0.318381,0.385286,0.280842,0.266102,0.335176,0.333381,0.333401,0.471013,0.362656,0.325470
Sambo Kojin,0.992000,0.953507,0.996462,0.996200,1.000000,0.985132,0.994712,0.984743,0.998826,0.955639,...,0.234619,0.303463,0.196196,0.181146,0.251855,0.250012,0.250031,0.392405,0.280122,0.241890


10. Recommendation Function for Cosine Similarity

In [10]:
def recommend_cosine(restaurant_name, similarity_matrix, top_n=3):
    if restaurant_name not in similarity_matrix.index:
        return f"Restaurant '{restaurant_name}' not found in dataset."

    similar_scores = similarity_matrix.loc[restaurant_name].sort_values(ascending=False)

    # Exclude the restaurant itself
    recommendations = similar_scores[1:top_n+1]
    return recommendations

11. Test Cosine Recommendations

In [11]:
recommend_cosine("Le Petit Souffle", df_cosine, top_n=3)

,Le Petit Souffle
Restaurant Name,
Cube - Tasting Kitchen,0.999997
Khidmat,0.999997
Chin Chin,0.999996


12. Show Recommended Restaurants from Original Data

In [12]:
sample_names = ['Le Petit Souffle', 'Cube - Tasting Kitchen', 'Khidmat']
df[df['Restaurant Name'].isin(sample_names)]

,Restaurant Name,City,Cuisines,Average Cost for two,Price range,Aggregate rating,Votes,Cuisines List
0,Le Petit Souffle,Makati City,"French, Japanese, Desserts",1100,3,4.8,314,"[French, Japanese, Desserts]"
4451,Khidmat,New Delhi,"North Indian, Mughlai, Chinese",1300,3,3.6,372,"[North Indian, Mughlai, Chinese]"
8729,Khidmat,Noida,"North Indian, Mughlai, Chinese",750,2,3.4,369,"[North Indian, Mughlai, Chinese]"
9457,Cube - Tasting Kitchen,Inner City,"European, Contemporary",1540,4,4.9,441,"[European, Contemporary]"


13. Scale Features Before Distance/Similarity Calculation

In [13]:
scaler = MinMaxScaler()

df_scaled = pd.DataFrame(
    scaler.fit_transform(df_full),
    index=df_full.index,
    columns=df_full.columns
)

df_scaled.head()

,Average Cost for two,Price range,Aggregate rating,Votes,Afghani,African,American,Andhra,Arabian,Argentine,...,Teriyaki,Tex-Mex,Thai,Tibetan,Turkish,Turkish Pizza,Vegetarian,Vietnamese,Western,World Cuisine
Restaurant Name,,,,,,,,,,,,,,,,,,,,,
Le Petit Souffle,0.001375,0.666667,0.979592,0.028718,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Izakaya Kikufuji,0.001500,0.666667,0.918367,0.054052,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Heat - Edsa Shangri-La,0.005000,1.000000,0.897959,0.024694,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Ooma,0.001875,1.000000,1.000000,0.033382,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Sambo Kojin,0.001875,1.000000,0.979592,0.020944,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


14. Cosine Similarity After Scaling

In [14]:
scaled_cosine_matrix = cosine_similarity(df_scaled)

df_scaled_cosine = pd.DataFrame(
    scaled_cosine_matrix,
    index=df_scaled.index,
    columns=df_scaled.index
)

df_scaled_cosine.head()

Restaurant Name,Le Petit Souffle,Izakaya Kikufuji,Heat - Edsa Shangri-La,Ooma,Sambo Kojin,Din Tai Fung,Buffet 101,Vikings,Spiral - Sofitel Philippine Plaza Manila,Locavore,...,Emirgan S�_ti��,Leman K�_lt�_r,Dem Karak�_y,Karak�_y G�_ll�_o��lu,Baltazar,Naml۱ Gurme,Ceviz A��ac۱,Huqqa,A���k Kahve,Walter's Coffee Roastery
Restaurant Name,,,,,,,,,,,,,,,,,,,,,
Le Petit Souffle,1.000000,0.738416,0.305882,0.630569,0.628959,0.420707,0.365105,0.299959,0.351024,0.431680,...,0.532486,0.397736,0.382546,0.591183,0.346827,0.411463,0.299599,0.354894,0.428009,0.365456
Izakaya Kikufuji,0.738416,1.000000,0.409258,0.854454,0.852428,0.559522,0.489459,0.401906,0.469099,0.573582,...,0.399148,0.530569,0.504762,0.420085,0.461901,0.548306,0.399279,0.476502,0.574158,0.482700
Heat - Edsa Shangri-La,0.305882,0.409258,1.000000,0.393924,0.392076,0.407564,0.592386,0.826522,0.723428,0.681484,...,0.291746,0.393276,0.343809,0.284781,0.336814,0.401844,0.291731,0.368680,0.440379,0.332056
Ooma,0.630569,0.854454,0.393924,1.000000,0.748719,0.521612,0.474493,0.388006,0.447431,0.531069,...,0.373014,0.501051,0.447877,0.371416,0.430901,0.513433,0.373019,0.464669,0.556220,0.431418
Sambo Kojin,0.628959,0.852428,0.392076,0.748719,1.000000,0.518046,0.472422,0.386182,0.445002,0.527144,...,0.370402,0.497987,0.443155,0.367273,0.427838,0.509929,0.370365,0.462776,0.553662,0.427122


15. Test Scaled Cosine Recommendations

In [15]:
recommend_cosine("Le Petit Souffle", df_scaled_cosine, top_n=3)

,Le Petit Souffle
Restaurant Name,
Tokyo Mon Amour,0.849186
Milse,0.738610
Izakaya Kikufuji,0.738416


In [25]:
sample_names = ['Le Petit Souffle', 'Tokyo Mon Amour', 'Milse']
df[df['Restaurant Name'].isin(sample_names)]

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
4993,18255171,Tokyo Mon Amour,1,New Delhi,"28, Main Market, Lodhi Colony, New Delhi",Lodhi Colony,"Lodhi Colony, New Delhi",0.000000,0.000000,"Japanese, French",...,Indian Rupees(Rs.),Yes,No,No,No,4,3.1,Orange,Average,10
9299,7001086,Milse,148,Auckland,"The Pavilions, 27 Tyler Street, Britomart, Auc...",Britomart,"Britomart, Auckland",174.768690,-36.844188,Desserts,...,NewZealand($),No,No,No,No,3,4.9,Dark Green,Excellent,754


# 16. Euclidean **Distance**

In [16]:
euclidean_matrix = pairwise_distances(df_scaled, metric="euclidean")

df_euclidean = pd.DataFrame(
    euclidean_matrix,
    index=df_scaled.index,
    columns=df_scaled.index
)

df_euclidean.head()

Restaurant Name,Le Petit Souffle,Izakaya Kikufuji,Heat - Edsa Shangri-La,Ooma,Sambo Kojin,Din Tai Fung,Buffet 101,Vikings,Spiral - Sofitel Philippine Plaza Manila,Locavore,...,Emirgan S�_ti��,Leman K�_lt�_r,Dem Karak�_y,Karak�_y G�_ll�_o��lu,Baltazar,Naml۱ Gurme,Ceviz A��ac۱,Huqqa,A���k Kahve,Walter's Coffee Roastery
Restaurant Name,,,,,,,,,,,,,,,,,,,,,
Le Petit Souffle,0.000000,1.415765,2.667921,1.763959,1.763851,2.001666,2.266743,2.669683,2.472317,2.000099,...,2.004407,2.012636,2.028924,1.766280,2.238973,2.005565,2.453433,2.272117,2.034859,2.034308
Izakaya Kikufuji,1.415765,0.000000,2.261062,1.057451,1.056388,1.414553,1.766796,2.261619,2.029241,1.415549,...,2.001108,1.423628,1.453050,1.765515,1.732720,1.416683,2.001348,1.771386,1.456822,1.456546
Heat - Edsa Shangri-La,2.667921,2.261062,0.000000,2.451632,2.450854,2.260788,2.001797,1.415294,1.735353,1.765889,...,2.667561,2.265394,2.333860,2.541098,2.472764,2.262107,2.667899,2.453917,2.238307,2.334951
Ooma,1.763959,1.057451,2.451632,0.000000,1.414415,1.766785,2.008466,2.453818,2.236198,1.764018,...,2.265770,1.780802,1.858070,2.110333,2.031807,1.771798,2.266113,2.015121,1.742453,1.865103
Sambo Kojin,1.763851,1.056388,2.450854,1.414415,0.000000,1.765749,2.006829,2.452891,2.236456,1.764052,...,2.264867,1.778244,1.857569,2.110580,2.031001,1.770349,2.265288,2.012948,1.740815,1.863384


17. Recommendation Function for Euclidean Distance

In [17]:
def recommend_euclidean(restaurant_name, distance_matrix, top_n=3):
    if restaurant_name not in distance_matrix.index:
        return f"Restaurant '{restaurant_name}' not found in dataset."

    distances = distance_matrix.loc[restaurant_name].sort_values()

    # Exclude the restaurant itself (distance = 0)
    recommendations = distances[1:top_n+1]
    return recommendations

18. Test Euclidean Recommendations

In [18]:
recommend_euclidean("Le Petit Souffle", df_euclidean, top_n=3)

,Le Petit Souffle
Restaurant Name,
Tokyo Mon Amour,1.110069
Milse,1.414934
Izakaya Kikufuji,1.415765


# 19. Manhattan Distance

In [19]:
manhattan_matrix = pairwise_distances(df_scaled, metric="manhattan")

df_manhattan = pd.DataFrame(
    manhattan_matrix,
    index=df_scaled.index,
    columns=df_scaled.index
)

df_manhattan.head()

Restaurant Name,Le Petit Souffle,Izakaya Kikufuji,Heat - Edsa Shangri-La,Ooma,Sambo Kojin,Din Tai Fung,Buffet 101,Vikings,Spiral - Sofitel Philippine Plaza Manila,Locavore,...,Emirgan S�_ti��,Leman K�_lt�_r,Dem Karak�_y,Karak�_y G�_ll�_o��lu,Baltazar,Naml۱ Gurme,Ceviz A��ac۱,Huqqa,A���k Kahve,Walter's Coffee Roastery
Restaurant Name,,,,,,,,,,,,,,,,,,,,,
Le Petit Souffle,0.000000,2.086683,7.422615,3.358906,3.341607,4.083770,5.516564,7.490107,6.387944,4.019938,...,4.175221,4.243325,4.436771,3.445701,5.154154,4.187483,6.189542,5.590721,4.551509,4.523239
Izakaya Kikufuji,2.086683,0.000000,5.386599,1.436010,1.428041,2.043980,3.442868,5.403423,4.423710,2.066746,...,4.088788,2.172439,2.350337,3.440901,3.067721,2.101050,4.103109,3.504288,2.465076,2.436805
Heat - Edsa Shangri-La,7.422615,5.386599,0.000000,6.113854,6.088507,5.343120,4.106997,2.080540,3.136643,3.442553,...,7.434571,5.502675,5.736937,6.827500,6.413504,5.446833,7.448892,6.183405,5.144193,5.782589
Ooma,3.358906,1.436010,6.113854,0.000000,2.032846,3.438651,4.198474,6.172017,5.029038,3.369515,...,5.524798,3.592902,3.786348,4.795278,4.503731,3.537060,5.539120,4.273632,3.234420,3.872816
Sambo Kojin,3.341607,1.428041,6.088507,2.032846,0.000000,3.425377,4.190505,6.164047,5.061885,3.361545,...,5.516828,3.584932,3.778378,4.787308,4.495761,3.529090,5.531150,4.265662,3.226450,3.864846


20. Recommendation Function for Manhattan Distance

In [20]:
def recommend_manhattan(restaurant_name, distance_matrix, top_n=3):
    if restaurant_name not in distance_matrix.index:
        return f"Restaurant '{restaurant_name}' not found in dataset."

    distances = distance_matrix.loc[restaurant_name].sort_values()

    # Exclude the restaurant itself (distance = 0)
    recommendations = distances[1:top_n+1]
    return recommendations

21. Test Manhattan Recommendations

In [21]:
recommend_manhattan("Le Petit Souffle", df_manhattan, top_n=3)

,Le Petit Souffle
Restaurant Name,
Tokyo Mon Amour,1.709450
Milse,2.061962
Izakaya Kikufuji,2.086683


# 22. Compare All Three Methods Together

In [22]:
restaurant_name = "Le Petit Souffle"

print("Top recommendations using Cosine Similarity:")
print(recommend_cosine(restaurant_name, df_scaled_cosine, top_n=3))

print("\nTop recommendations using Euclidean Distance:")
print(recommend_euclidean(restaurant_name, df_euclidean, top_n=3))

print("\nTop recommendations using Manhattan Distance:")
print(recommend_manhattan(restaurant_name, df_manhattan, top_n=3))

Top recommendations using Cosine Similarity:
Restaurant Name
Tokyo Mon Amour     0.849186
Milse               0.738610
Izakaya Kikufuji    0.738416
Name: Le Petit Souffle, dtype: float64

Top recommendations using Euclidean Distance:
Restaurant Name
Tokyo Mon Amour     1.110069
Milse               1.414934
Izakaya Kikufuji    1.415765
Name: Le Petit Souffle, dtype: float64

Top recommendations using Manhattan Distance:
Restaurant Name
Tokyo Mon Amour     1.709450
Milse               2.061962
Izakaya Kikufuji    2.086683
Name: Le Petit Souffle, dtype: float64
